# Initialization

In [1]:
import json
import uuid
import os
import json
from dotenv import load_dotenv
from pathlib import Path
from kafka import KafkaProducer
from faker import Faker
from time import sleep

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Ruang Data Project Spark-Kafka") 
    .config("spark.streaming.stopGracefullyOnShutdown", True) 
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]") 
    .getOrCreate()
)

spark

In [9]:
streaming = (
    spark
    .readStream
    .schema(dataSchema)
    .option('maxFilesPerTrigger', 1)
    .json('/resources/data/activity-data/')
)

In [10]:
# set partitions
spark.conf.set('spark.sql.shuffle.partitions', 5)

In [11]:
activityCounts = streaming.select('index').distinct()
activityQuery = (
    activityCounts.writeStream
    .queryName('activity_counts_3')
    .format('memory')
    .outputMode('append')
    .start()
)

# activityQuery.awaitTermination()

In [13]:
# activityQuery.awaitTermination()
activityQuery.stop()

In [12]:
from time import sleep
for x in range(5):
    spark.sql("SELECT COUNT(*) FROM activity_counts_3").show()
    sleep(1)

+--------+
|count(1)|
+--------+
|  290684|
+--------+

+--------+
|count(1)|
+--------+
|  320527|
+--------+

+--------+
|count(1)|
+--------+
|  340258|
+--------+

+--------+
|count(1)|
+--------+
|  353626|
+--------+

+--------+
|count(1)|
+--------+
|  362586|
+--------+



# Spark - Kafka Streaming

In [3]:
dotenv_path = Path('/resources/.env')
load_dotenv(dotenv_path=dotenv_path)

True

In [4]:
kafka_host = os.getenv('KAFKA_HOST')
kafka_topic = os.getenv('KAFKA_TOPIC_NAME')
kafka_topic_partition = os.getenv('KAFKA_TOPIC_NAME')+"-1"

In [5]:
kafka_topic

'test-topic-1'

## Batch Simulation

In [6]:
kafka_df = (
    spark
    .read
    .format("kafka")
    .option("kafka.bootstrap.servers", f'{kafka_host}:9092')
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .load()
)

In [7]:
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [8]:
kafka_df.show()

+----+--------------------+------------+---------+------+--------------------+-------------+
| key|               value|       topic|partition|offset|           timestamp|timestampType|
+----+--------------------+------------+---------+------+--------------------+-------------+
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     0|2025-02-27 04:55:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     1|2025-02-27 04:55:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     2|2025-02-27 04:56:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     3|2025-02-27 04:56:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     4|2025-02-27 04:56:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     5|2025-02-27 04:56:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     6|2025-02-27 04:56:...|            0|
|null|[7B 22 65 6D 70 5...|test-topic-1|        0|     7|2025-02-27 04

In [9]:
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))

In [10]:
kafka_json_df.show(5)

+----+--------------------+------------+---------+------+--------------------+-------------+
| key|               value|       topic|partition|offset|           timestamp|timestampType|
+----+--------------------+------------+---------+------+--------------------+-------------+
|null|{"emp_id": "a69b3...|test-topic-1|        0|     0|2025-02-27 04:55:...|            0|
|null|{"emp_id": "88e5e...|test-topic-1|        0|     1|2025-02-27 04:55:...|            0|
|null|{"emp_id": "ff137...|test-topic-1|        0|     2|2025-02-27 04:56:...|            0|
|null|{"emp_id": "89b52...|test-topic-1|        0|     3|2025-02-27 04:56:...|            0|
|null|{"emp_id": "98132...|test-topic-1|        0|     4|2025-02-27 04:56:...|            0|
+----+--------------------+------------+---------+------+--------------------+-------------+
only showing top 5 rows



In [11]:
(
    kafka_json_df
    .select('value')
    .limit(5)
    .collect()
)

[Row(value='{"emp_id": "a69b3ce9-586d-4e1c-9d0a-7b4a9db036e8", "employee_name": "Jessica Brewer", "department": "Marketing", "state": "RJ", "salary": 94646, "age": 18, "bonus": 27519, "ts": 1571744858}'),
 Row(value='{"emp_id": "88e5e6e3-1893-4d31-b9b7-6e2a626ac8e7", "employee_name": "Nicole Thompson", "department": "HR", "state": "TX", "salary": 95602, "age": 58, "bonus": 38802, "ts": 687804836}'),
 Row(value='{"emp_id": "ff137221-08f7-4a14-a6ef-da525fabe750", "employee_name": "Stephanie Mcdonald", "department": "Sales", "state": "CA", "salary": 134020, "age": 59, "bonus": 52695, "ts": 661254400}'),
 Row(value='{"emp_id": "89b52a4a-1544-4286-abf9-bc402c3d5579", "employee_name": "Olivia Allen", "department": "Sales", "state": "RJ", "salary": 82471, "age": 35, "bonus": 35968, "ts": 1524139641}'),
 Row(value='{"emp_id": "98132152-5cde-4be6-b9bf-eb8df533da45", "employee_name": "Kyle Smith", "department": "Marketing", "state": "CA", "salary": 109935, "age": 42, "bonus": 25813, "ts": 935979

In [12]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

schema = StructType(
    [
        StructField("emp_id", StringType(), True),
        StructField("employee_name", StringType(), True),
        StructField("department", StringType(), True),
        StructField("state", StringType(), True),
        StructField("salary", LongType(), True),
        StructField("age", IntegerType(), True),
        StructField("bonus", LongType(), True),
        StructField("ts", LongType(), True),
    ]
)

In [13]:
from pyspark.sql.functions import from_json, col

(
    kafka_json_df
    .select(
        from_json(col("value"), schema)
        .alias("data")
    )
    .select("data.*")
    .show()
)

+--------------------+------------------+----------+-----+------+---+-----+----------+
|              emp_id|     employee_name|department|state|salary|age|bonus|        ts|
+--------------------+------------------+----------+-----+------+---+-----+----------+
|a69b3ce9-586d-4e1...|    Jessica Brewer| Marketing|   RJ| 94646| 18|27519|1571744858|
|88e5e6e3-1893-4d3...|   Nicole Thompson|        HR|   TX| 95602| 58|38802| 687804836|
|ff137221-08f7-4a1...|Stephanie Mcdonald|     Sales|   CA|134020| 59|52695| 661254400|
|89b52a4a-1544-428...|      Olivia Allen|     Sales|   RJ| 82471| 35|35968|1524139641|
|98132152-5cde-4be...|        Kyle Smith| Marketing|   CA|109935| 42|25813| 935979392|
|e9658aa3-0ea2-439...|      Cathy Martin| Marketing|   IL| 14193| 19|37039|1489382320|
|c5f92c15-c739-4c7...|  Michael Ferguson| Marketing|   IL| 68350| 49|14449|1699697388|
|fc3f8abf-1556-467...|     Daniel Martin| Marketing|   CA|147579| 59|57695| 376308994|
|04cb8fd0-d74e-48d...|       Ryan Guzman|  

## Stream Simulation

In [14]:
kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f'{kafka_host}:9092')
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .load()
)

In [15]:
from pyspark.sql.functions import from_json, col

parsed_df = (
    kafka_df
    .withColumn("value", expr("cast(value as string)"))
    .select(
        from_json(col("value"), schema)
        .alias("data")
    )
    .select("data.*")
)

In [ ]:
print_console = parsed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(processingTime='5 seconds') \
    .start()
print_console.awaitTermination()

In [ ]:
(
    parsed_df
    .writeStream
    .format("console")
    .outputMode("append")
    .trigger(processingTime='5 seconds')
    # .trigger(continuous='1 second')
    # .trigger(once=true)
    .option("checkpointLocation", "checkpoint_dir")
    .start()
    .awaitTermination()
)